In [ ]:
import pandas as pd
from transformers import AutoTokenizer, AutoModel
import numpy as np
import string
import torch
from torch.nn.utils.rnn import pad_sequence

df = pd.read_csv("datasets/svo_word_level.csv")

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
model = (
    AutoModel.from_pretrained(
        "distilbert-base-uncased", output_hidden_states=True
    )
    .cuda()
    .eval()
)

In [ ]:
words = df[["word", "sentence_id"]]
words

In [ ]:
def cum_join_index(words):
    s = ""
    idx = []
    for w in words:
        if w in string.punctuation:
            s = s.strip()
        idx.append(len(s))
        s += w + " "

    return s.strip(), idx

In [ ]:
sentences = words.groupby("sentence_id").word.apply(cum_join_index)

In [ ]:
sentences = sentences.apply(pd.Series).reset_index(
    names=["sentence_id", "sentence", "word_start_index"]
)
sentences

In [ ]:
word_start_index = [
    torch.Tensor(i) for i in sentences.word_start_index.tolist()
]
word_start_index = pad_sequence(word_start_index, batch_first=True)

In [ ]:
inputs = tokenizer(
    sentences.sentence.tolist(),
    return_offsets_mapping=True,
    return_tensors="pt",
    padding=True,
    truncation=False,
)
# Get start and end index in original string for each token
offset_mapping = inputs.pop("offset_mapping")

In [ ]:
with torch.no_grad():
    hidden_states = model(**inputs.to("cuda")).hidden_states
    hidden_states = torch.stack(hidden_states, dim=-2)

In [ ]:
special_tokens = offset_mapping[:, :, 1] == offset_mapping[:, :, 0]

In [ ]:
i_leq_a = offset_mapping[:, None, :, 0] <= word_start_index[:, :, None]
b_l_j = (
    word_start_index.roll(shifts=-1)[:, :, None] < offset_mapping[:, None, :, 1]
)

In [ ]:
offset_mapping[0]

In [ ]:
b_l_j[0]

In [ ]:
i_leq_a

In [ ]:
offset_mapping[-1]

In [ ]:
word_start_index.roll(shifts=-1)

In [ ]:
b_l_j

In [ ]:
token_word_belong = 

In [ ]:
(offset_mapping[:, None, :, 0] <= word_start_index[:, :, None]).shape

In [ ]:
def make_sentence(words):
    s = ""
    cum_s = []
    indices = []
    for word in words:
        if word in string.punctuation:
            s = s.strip()
        indices.append(len(s))
        s += word
        cum_s.append(s)
        s += " "

    return (s.strip(), indices, cum_s)

In [ ]:
def events_from_words(words: pd.DataFrame) -> pd.DataFrame:
    events = words.copy()
    events["sentence"] = events.groupby("sentence_id").word.transform(
        lambda words: make_sentence(words)[0]
    )
    events["sentence_char"] = events.groupby("sentence_id").word.transform(
        lambda words: make_sentence(words)[1]
    )
    events["context"] = events.groupby("sentence_id").word.transform(
        lambda words: make_sentence(words)[2]
    )
    events = events.rename(columns={"word": "text"})
    events["timeline"] = "svo_word_level"
    events["language"] = "en"
    events["type"] = "Word"
    events["start"] = events.index
    events["duration"] = 0.5
    events = ns.segments.validate_events(events)

    return events

In [ ]:
events = events_from_words(words)
events

In [ ]:
feature = HuggingFaceText(
    contextualized=True,
    token_aggregation="mean",
    model_name="gpt2",
    device="cuda",
    batch_size=64,
    layers=2 / 3,
    cache_all_layers=True,
)

In [ ]:
events = feature._events_from_dataframe(events)

In [ ]:
data = feature._get_data([row for _, row in events.iterrows()])
data = list(data)
data = np.array(data)
data = torch.Tensor(data)
data.shape

In [ ]:
data = list(
    feature._get_timed_arrays(
        events,
        start=0,
        duration=150,
    )
)

In [ ]:
data[0].data